In [8]:
import pandas as pd
import numpy as np
import requests
import folium
from folium.plugins import HeatMap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [9]:
# LOAD DATASET FOR ML TRAINING


data = pd.read_csv(r"C:\Users\iamsu\Downloads\AI-BASED-EARLY-WARNING-SYSTEM\AI-BASED-EARLY-WARNING-SYSTEM\data\shipment_dataset_odisha_modified.csv").ffill().bfill()

le_origin = LabelEncoder()
le_dest = LabelEncoder()
le_weather = LabelEncoder()
le_traffic = LabelEncoder()
le_carrier = LabelEncoder()

data["origin"] = le_origin.fit_transform(data["origin"].astype(str))
data["destination"] = le_dest.fit_transform(data["destination"].astype(str))
data["weather"] = le_weather.fit_transform(data["weather"].astype(str))
data["traffic"] = le_traffic.fit_transform(data["traffic"].astype(str))
data["Carrier_History"] = le_carrier.fit_transform(data["Carrier_History"].astype(str))

X = data.drop(["shipment_id","delay"],axis=1).astype(np.float32)
y = data["delay"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = XGBClassifier(n_estimators=200,learning_rate=0.6,max_depth=12,subsample=0.9,colsample_bytree=0.9,random_state=42,eval_metric="logloss")
model.fit(X_train,y_train)

print("Model Accuracy:",accuracy_score(y_test,model.predict(X_test)))

Model Accuracy: 0.88


In [10]:
# CITY COORDINATES


city_coords = {

"Delhi": (28.6139,77.2090),
"Mumbai": (19.0760,72.8777),
"Bangalore": (12.9716,77.5946),
"Chennai": (13.0827,80.2707),
"Hyderabad": (17.3850,78.4867),
"Pune": (18.5204,73.8567),
"Ahmedabad": (23.0225,72.5714),
"Kolkata": (22.5726,88.3639),
"Lucknow": (26.8467,80.9462),
"Jaipur": (26.9124,75.7873),
"Puri": (19.8135,85.8312),
"Bhubaneswar": (20.2961,85.8245),
"Cuttack": (20.4625,85.8828),
"Baripada": (21.9333,86.7500),
"Nagpur": (21.1458,79.0882),
"Angul": (20.8399,85.1018),
"Rourkela": (22.2604,84.8536),
"Keonjhar": (21.6231,85.5994),
"Dhenkanal": (20.6602,85.5994),
"Balasore": (21.4938,86.9311),
"Sambalpur": (21.4667,83.9667),
"Berhampur": (19.3144,84.7911),
"Patna": (25.5941,85.1376),
"Varanasi": (25.3176,82.9739),
"Vishakhapatnam": (17.6868,83.2185),
"Jharsuguda": (21.9100,84.0000),
"Raipur": (21.2514,81.6296),
"Baripada": (21.9333,86.7500),
"Ranch": (23.3441,85.3096),

}

In [11]:
# WEATHER FUNCTION

def weather_level(city):

    url=f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_KEY}"

    try:
        r=requests.get(url,timeout=3).json()
        w=r["weather"][0]["main"]
    except:
        return 0

    if w in ["Rain","Thunderstorm"]:
        return 2
    elif w=="Clouds":
        return 1
    else:
        return 0

In [12]:
# TRAFFIC FUNCTION

def traffic_penalty(lat,lon):

    url=f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json?point={lat},{lon}&key={TOMTOM_KEY}"

    try:

        r=requests.get(url,timeout=3).json()
        flow=r["flowSegmentData"]

        ratio=flow["currentSpeed"]/flow["freeFlowSpeed"]

        if ratio<0.5:
            return 2
        elif ratio<0.8:
            return 1

    except:
        pass

    return 0